# A04 · 搜索广告匹配与质量排序

先修：公用31–34；A03。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

自然检索回答信息需求；广告候选还必须满足活动状态、地域、物料与预算资格。此处关键词OR触发，地域与状态AND过滤，质量×CPC报价决定排序分。质量是构造值，不是从匿名Criteo字段推导的落地页评分；真实系统还需要审核和落地页一致性。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
pool=ac.candidates();display(pd.DataFrame([{k:a[k] for k in ('id','quality','bid')} for a in pool]))
assert [a['id'] for a in pool]==['a','b']
print('北京', [a['id'] for a in ac.candidates(region='北京')])
print('未匹配查询',ac.candidates(query='药品'))
organic=['跑鞋选购知识','运动损伤预防'];print('自然结果',organic,'商业候选',[a['id'] for a in pool])

,id,quality,bid
0,a,0.8,2.0
1,b,0.5,2.5


北京 ['b']
未匹配查询 []
自然结果 ['跑鞋选购知识', '运动损伤预防'] 商业候选 ['a', 'b']


## 解释与局限

最高原始报价并未获最高排序，且未启用的c即使质量和报价高也不得展示。自然结果列表独立，不能把文档ID当广告活动。地域只改一个变量，资格结果随之变化。

## 查看源码，再做练习

实现短语匹配和否定关键词；用“便宜跑鞋”和“二手跑鞋”对照，检查硬约束是否在排序之前生效。

In [3]:
import inspect
print(inspect.getsource(ac.candidates))

def candidates(query='跑鞋', region='上海', topk=3):
    ads=inventory(); index={}
    for a in ads:
        for key in a['keywords']: index.setdefault(key,set()).add(a['id'])
    tokens=set(query.split()); recalled=set().union(*(index.get(t,set()) for t in tokens))
    eligible=[a for a in ads if a['id'] in recalled and region in a['regions'] and a['active'] and a['budget']>0]
    oracle=[a for a in ads if tokens & a['keywords'] and region in a['regions'] and a['active'] and a['budget']>0]
    assert {a['id'] for a in eligible}=={a['id'] for a in oracle}
    return sorted(eligible,key=lambda a:(-a['quality']*a['bid'],a['id']))[:topk]



## 来源与下一步

A-CCR 18–20、39页；A-NJU搜索市场主题。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。